# Aula 3, ChromaDB na prática

Notebook executável que acompanha a aula [03-chromadb-na-pratica.md](../../lessons/modulo-09-rag/03-chromadb-na-pratica.md).

## O que vamos fazer aqui

Um banco vetorial guarda vetores e busca os mais parecidos com uma consulta. Vamos construir um
banco vetorial mínimo em memória, com uma API parecida com a do ChromaDB, e depois ver o ChromaDB
de verdade como caminho opcional. Python puro.

## Um banco vetorial mínimo

A classe encapsula indexar e consultar, exatamente o contrato que um banco vetorial oferece.

In [ ]:
import re
import math
from collections import Counter


class BancoVetorialMinimo:
    """Um banco vetorial em memória, com TF-IDF e busca por cosseno."""

    def __init__(self):
        self.textos = []
        self.vetores = []
        self.idf = {}

    def indexar(self, documentos):
        self.textos = list(documentos)
        n = len(documentos)
        df = Counter()
        for d in documentos:
            for w in set(self._tok(d)):
                df[w] += 1
        self.idf = {w: math.log(n / f) for w, f in df.items()}
        self.vetores = [self._vetor(d) for d in documentos]

    def consultar(self, pergunta, k=2):
        q = self._vetor(pergunta)
        ranking = sorted(
            ((self._cos(q, self.vetores[i]), i) for i in range(len(self.textos))),
            reverse=True,
        )
        return [(round(s, 3), self.textos[i]) for s, i in ranking[:k]]

    def _tok(self, texto):
        return re.findall(r"\w+", texto.lower())

    def _vetor(self, texto):
        tf = Counter(self._tok(texto))
        return {w: tf[w] * self.idf.get(w, 0.0) for w in tf}

    def _cos(self, a, b):
        prod = sum(a[w] * b.get(w, 0.0) for w in a)
        na = math.sqrt(sum(v * v for v in a.values()))
        nb = math.sqrt(sum(v * v for v in b.values()))
        return prod / (na * nb) if na and nb else 0.0


banco = BancoVetorialMinimo()
banco.indexar([
    "A derivada mede a taxa de variação de uma função.",
    "Uma matriz organiza números em linhas e colunas.",
    "Em Python, def serve para definir uma função.",
])
for score, texto in banco.consultar("o que é a derivada?", k=2):
    print(score, texto)

O banco recupera primeiro o trecho da derivada, com uma API limpa de indexar e
consultar. É esse mesmo contrato que o ChromaDB oferece, com persistência e busca eficiente.

## Caminho opcional, o ChromaDB de verdade

Faz o mesmo trabalho em poucas linhas. Roda apenas se o chromadb estiver instalado.

In [ ]:
try:
    import chromadb

    cliente = chromadb.Client()
    colecao = cliente.create_collection(name="aulas")
    colecao.add(
        documents=[
            "A derivada mede a taxa de variação de uma função.",
            "Uma matriz organiza números em linhas e colunas.",
            "Em Python, def serve para definir uma função.",
        ],
        ids=["d1", "d2", "d3"],
    )
    res = colecao.query(query_texts=["o que é a derivada?"], n_results=2)
    for doc in res["documents"][0]:
        print(doc)
except Exception as erro:
    print("chromadb não disponível; o banco mínimo acima já demonstra o conceito.")
    print("Detalhe:", erro)

O ChromaDB gera os embeddings, guarda tudo e busca, com uma API enxuta. Para o projeto,
encapsule o armazenamento atrás de uma interface, podendo trocar o banco mínimo pelo ChromaDB.